# Blok 1 · Zadanie grupowe: kto anuluje rezerwację? 🏨

**Sytuacja:** jesteście zespołem data science sieci hoteli w Portugalii. Anulowane rezerwacje to realne straty —
puste pokoje, chybiony overbooking, źle zaplanowany personel. Revenue manager prosi o model, który
**w momencie przyjęcia rezerwacji** przewidzi, czy zostanie ona anulowana.

Target `is_canceled` ma dwie klasy (0/1) → to **klasyfikacja**. Jakim modelem? **To wasza decyzja**

To ten sam workflow, który przeszliśmy na demo — ale dane są **brudne**: mają braki, dziesiątki kolumn różnych
typów i pułapki. Tak wyglądają prawdziwe projekty. (Dane są prawdziwe — pochodzą z systemów rezerwacyjnych
dwóch hoteli, opisanych w publikacji Antonio, de Almeida & Nunes, 2019.)

## Zasady

* Pracujecie indywidualnie lub w grupach 2–3 osobowych, macie **ok. 50 minut**.
* Przechodzicie przez kroki 1–8 poniżej. Komórki z `# TODO` uzupełniacie sami.

## Dane — 27 cech + target

Nie musicie użyć wszystkich kolumn! Wybierzcie sensowny podzbiór i **iterujcie**.

| kolumna | znaczenie |
|---|---|
| `is_canceled` | **TARGET**: 1 = rezerwacja anulowana |
| `hotel` | typ hotelu: Resort Hotel / City Hotel |
| `lead_time` | ile dni przed przyjazdem złożono rezerwację |
| `arrival_date_year/month/week_number/day_of_month` | data przyjazdu (4 kolumny) |
| `stays_in_weekend_nights` / `stays_in_week_nights` | liczba nocy weekendowych / roboczych |
| `adults`, `children`, `babies` | liczba gości |
| `meal` | wykupione wyżywienie (BB/HB/FB/SC) |
| `country` | kraj pochodzenia gościa (kod ISO) |
| `market_segment` | segment rynku (Online TA, Groups, Direct...) |
| `distribution_channel` | kanał dystrybucji (TA/TO, Direct, GDS...) |
| `is_repeated_guest` | czy gość powracający (0/1) |
| `previous_cancellations` | ile rezerwacji ten gość wcześniej anulował |
| `previous_bookings_not_canceled` | ile wcześniej zrealizował |
| `reserved_room_type` | typ zarezerwowanego pokoju |
| `deposit_type` | rodzaj depozytu (No Deposit / Non Refund / Refundable) |
| `agent` / `company` | ID biura podróży / firmy, przez które złożono rezerwację |
| `days_in_waiting_list` | dni na liście oczekujących |
| `customer_type` | typ klienta (Transient, Contract, Group...) |
| `adr` | średnia cena za dobę (Average Daily Rate) |
| `required_car_parking_spaces` | liczba miejsc parkingowych |
| `total_of_special_requests` | liczba specjalnych życzeń |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
# wczytanie danych
import os

df = pd.read_csv("https://raw.githubusercontent.com/ml-course-pg/ml-course-pg-2026/main/01-wprowadzenie/data/hotels.csv")

print("wymiary:", df.shape)
df.head()

## Krok 1: EDA — poznajcie swoje dane (~5 min)

Zanim cokolwiek zbudujecie, odpowiedzcie na trzy pytania:

a. **Które kolumny mają braki i ile procent?**

b. **Czy klasy targetu są zbalansowane?**

c. **Które cechy najmocniej wiążą się z anulowaniem?**

In [ ]:
# TODO 1a

# TODO 1b

# TODO 1c

## Krok 2: Decyzje o cechach (~5 min)

Zdecydujcie, **które kolumny wchodzą do modelu** i podzielcie je na numeryczne / kategoryczne.

Pytania-pułapki (przedyskutujcie w grupie!):

* jakie kolumny należy wyrzucić? imputować? procesować?
* jakie liczby to szum, a jakie to faktyczna wartość?

In [ ]:
# TODO 2: wybierzcie kolumny do modelu
cechy_numeryczne   = [...] 
cechy_kategoryczne = [...] 

## Krok 3: Podział train / walidacja (kroki 3–6 razem: ~15 min)

Potrzebujecie własnej **walidacji**, żeby podejmować decyzje.

> `stratify=y` zapewnia ten sam odsetek anulowań w obu częściach — przy klasyfikacji zawsze warto.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[cechy_numeryczne + cechy_kategoryczne]
y = df["is_canceled"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print(f"train: {len(X_train)}, walidacja: {len(X_val)}")

## Krok 4: Preprocessing w pipeline

Cechy numeryczne i kategoryczne wymagają **innego** przetwarzania. Do tego służy `ColumnTransformer` — kierownik ruchu, który każdą grupę kolumn wysyła do innego mini-pipeline'u:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# TODO 4: uzupełnijcie oba mini-pipeline'y
pipe_num = Pipeline([
    ("imputacja", ...),      # czym uzupełnić braki w liczbach?
    ("skalowanie", ...),
])

pipe_kat = Pipeline([
    ("imputacja", ...),      # czym uzupełnić braki w kategoriach?
    ("kodowanie", ...),
])

preprocessing = ColumnTransformer([
    ("numeryczne", pipe_num, cechy_numeryczne),
    ("kategoryczne", pipe_kat, cechy_kategoryczne),
])
preprocessing

## Krok 5: Baseline

Jak na demo: zanim ocenicie swój model, sprawdźcie **najbardziej podstawowy** model

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
acc_baseline = accuracy_score(y_val, baseline.predict(X_val))
print(f"baseline (zawsze 'nie anuluje'): accuracy = {acc_baseline:.3f}")

## Krok 6: Właściwy model — wasza decyzja

Wybierzcie klasyfikator ze scikit-learn, który uważacie za odpowiedni, i **umiejcie uzasadnić wybór**.
Możecie też wytrenować kilka i porównać na walidacji.

> Do sekcji KPI na końcu przyda się model z metodą `predict_proba` (większość klasyfikatorów sklearn ją ma).

In [ ]:
# TODO 6: zaimportujcie WYBRANY przez was klasyfikator, sklejcie go z preprocessingiem
#         w jeden pipeline i oceńcie na walidacji
model = Pipeline([
    ("preprocessing", ...),
    ("klasyfikator", ...),
])

# model.fit(...)
# acc = ...
# print(f"regresja logistyczna: accuracy = {acc:.3f}  (baseline: {acc_baseline:.3f})")

## Krok 7: Iteracja — poprawcie wynik! (~20 min)

Macie działający model. Teraz spróbujcie go ulepszyć — **każdą zmianę oceniajcie na walidacji**. Pomysły:

* dodajcie kolumny, które odrzuciliście na starcie 
* zmieńcie strategię imputacji
* regularyzacja — jeśli wybraliście model liniowy (np. regresję logistyczną)
* ...albo w ogóle inny klasyfikator — porównajcie na walidacji.

In [ ]:
# TODO 7: miejsce na wasze eksperymenty 

## Krok 8: Zapis modelu 📦

Zapiszcie **cały pipeline** (preprocessing + model). Zmieńcie `NUMER_GRUPY`!

In [ ]:
import joblib

NUMER_GRUPY = "XX"

nazwa_pliku = f"grupa_{NUMER_GRUPY}.joblib"
joblib.dump(model, nazwa_pliku)
print(f"zapisano: {nazwa_pliku}")

---

## 💰 Ile ten model jest wart? KPI biznesowe

Accuracy nie przekona zarządu — **euro przekonają**. Policzmy, ile model realnie zarabia, przy jawnych założeniach.

**Scenariusz użycia:** gdy model przewiduje anulowanie, hotel **odsprzedaje pokój** (kontrolowany overbooking).

**Założenia:**

| zdarzenie | skutek | przyjęta wartość |
|---|---|---|
| wartość rezerwacji | `adr` × liczba nocy | mediana ze zbioru |
| **TP** — przewidział anulację, gość faktycznie anulował | pokój odsprzedany last-minute (z rabatem) | **+60%** wartości rezerwacji |
| **FP** — przewidział anulację, gość przyjechał | overbooking: relokacja gościa + rekompensata + reputacja | **−150%** wartości rezerwacji |
| **FN** — nie przewidział, gość anulował | pusty pokój — tak samo jak bez modelu | 0 (nic nie zmieniamy) |
| **TN** — nie przewidział, gość przyjechał | normalna doba hotelowa | 0 |

Punkt odniesienia to hotel **bez modelu** (zysk dodatkowy = 0). Kluczowa obserwacja: model nie musi przewidywać przy progu 0.5!
`predict_proba` daje prawdopodobieństwo — **próg decyzyjny to parametr biznesowy**, który wybieramy tak, żeby maksymalizować zysk, a nie accuracy.

In [ ]:
# --- założenia biznesowe (zmieńcie i zobaczcie, co się dzieje!) ---
WARTOSC_REZERWACJI = float((df["adr"] * (df["stays_in_week_nights"] + df["stays_in_weekend_nights"])).median())
ZYSK_TP  = 0.60 * WARTOSC_REZERWACJI    # odsprzedaż z rabatem
KOSZT_FP = 1.50 * WARTOSC_REZERWACJI    # relokacja + rekompensata + reputacja
REZERWACJI_ROCZNIE = 50_000             # skala biznesu obu hoteli

print(f"mediana wartości rezerwacji: {WARTOSC_REZERWACJI:.0f} €")

def zysk_modelu(model, X, y, prog=0.5):
    """dodatkowy zysk [€ na rezerwację] względem hotelu bez modelu"""
    p_anulowania = model.predict_proba(X)[:, 1]
    dzialamy = p_anulowania >= prog          # odsprzedajemy pokój
    tp = ((dzialamy) & (y == 1)).sum()
    fp = ((dzialamy) & (y == 0)).sum()
    return (tp * ZYSK_TP - fp * KOSZT_FP) / len(y)

# zysk przy naiwnym progu 0.5 vs progi dobrane biznesowo
progi = np.linspace(0.05, 0.95, 19)
zyski = [zysk_modelu(model, X_val, y_val, prog=p) for p in progi]
najlepszy_prog = progi[int(np.argmax(zyski))]

plt.plot(progi, zyski, lw=2)
plt.axhline(0, color="gray", ls=":")
plt.axvline(0.5, color="crimson", ls="--", label=f"próg 0.5: {zysk_modelu(model, X_val, y_val, 0.5):.2f} €/rez.")
plt.axvline(najlepszy_prog, color="tab:green", ls="--", label=f"próg {najlepszy_prog:.2f}: {max(zyski):.2f} €/rez.")
plt.xlabel("próg decyzyjny (od jakiego P(anulowanie) odsprzedajemy pokój)")
plt.ylabel("dodatkowy zysk [€ na rezerwację]")
plt.title("Wartość biznesowa modelu w funkcji progu decyzyjnego")
plt.legend(); plt.show()

for prog in [0.5, najlepszy_prog]:
    z = zysk_modelu(model, X_val, y_val, prog)
    print(f"próg {prog:.2f}:  {z:+.2f} € na rezerwację  =>  {z * REZERWACJI_ROCZNIE / 1000:+,.0f} tys. € rocznie")

### Co z tego wynika — i jak używać modelu w praktyce

Trzy wnioski z wykresu (przedyskutujcie w grupie):

1. **Optymalny próg ≠ 0.5.** Skoro FP kosztuje 2.5× więcej niż zarabia TP, opłaca się działać tylko przy dużej pewności — matematycznie: gdy `p × 60% > (1−p) × 150%`, czyli p > ~0.71. Model z najlepszym accuracy może zarabiać **mniej** niż model gorszy, ale lepiej "wycelowany".
2. **Zmieńcie założenia** (np. tańsza rekompensata, mniejszy rabat odsprzedaży) i zobaczcie, jak wędruje optymalny próg — to pokazuje, że wdrożenie ML to decyzja biznesowa, nie tylko techniczna.

**Jak taki model działa w prawdziwym hotelu:**

* **Kontrolowany overbooking** — codzienny raport: suma P(anulowanie) na najbliższe daty mówi, ile pokoi można "dosprzedać" ponad stan.
* **Retencja wysokiego ryzyka** — zamiast odsprzedawać, można do rezerwacji z p > progu zadzwonić. Inne koszty FP → inny optymalny próg, ta sama maszyneria.
* **Polityka depozytów** — segmenty o wysokim ryzyku dostają przy rezerwacji wymóg przedpłaty; model wskazuje, które.
* **Planowanie operacyjne** — prognoza realnego obłożenia (rezerwacje minus przewidziane anulacje) zasila grafiki personelu i zakupy.

Wspólny mianownik: model **nie podejmuje decyzji** — dostarcza prawdopodobieństwo, a decyzję (i próg) ustawia biznes, znając swoje koszty.